In [ ]:
# Cell 1: check GPU
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import torch
print(torch.cuda.is_available())   # should be True if you choose option 1

ModuleNotFoundError: No module named 'torch'

In [16]:
!rm -rf /content/HYPER

In [20]:
!mkdir /content/HYPER

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


In [22]:
# Cell 2
!git clone https://github.com/behnooshbhy2002/HyperTextual.git /content/HYPER
%cd /content/HYPER
!ls

Cloning into '/content/HYPER'...
remote: Enumerating objects: 336, done.
remote: Counting objects: 100% (336/336), done.
remote: Compressing objects: 100% (244/244), done.
remote: Total 336 (delta 95), reused 329 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (336/336), 29.03 MiB | 15.74 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/HYPER
ckpts		 files		     hyper_text  requirements.txt
config		 hyper		     LICENSE	 script
data_generation  hypergraph_dataset  README.md


In [4]:
# Cell 3
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())
TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '')
print(f"Wheel index: https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html")

2.11.0+cu128 12.8 True
Wheel index: https://data.pyg.org/whl/torch-2.11.0+cu128.html


In [5]:
# Cell 4
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install torch-geometric
!pip install ninja easydict pyyaml neptune jinja2 tqdm

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 119.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.8/487.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.7 MB/s eta 0:00:00


In [6]:
# Cell 5 — confirm Triton works (this is the single biggest VRAM factor — see below)
import triton
print("triton OK:", triton.__version__)

triton OK: 3.6.0


In [7]:
# Cell — fix working directory
import os
os.chdir('/content/HYPER')
print(os.getcwd())
print(os.path.exists('hypergraph_dataset/JF-IND/train.txt'))  # باید True بشه

/content/HYPER
True


In [8]:
# Cell — fix the bug in run.py
bug_line = """            val_filtered_data = Data(
                edge_index=torch.cat([train_data.edge_index, valid_data.target_edge_index], dim=1),
                edge_type=torch.cat([train_data.edge_type, valid_data.target_edge_type], num_nodes = valid_data.num_nodes)
            )"""

fixed_line = """            val_filtered_data = Data(
                edge_index=torch.cat([train_data.edge_index, valid_data.target_edge_index], dim=1),
                edge_type=torch.cat([train_data.edge_type, valid_data.target_edge_type]),
                num_nodes=valid_data.num_nodes
            )"""

with open('/content/HYPER/script/run.py', 'r') as f:
    content = f.read()

content = content.replace(bug_line, fixed_line)

with open('/content/HYPER/script/run.py', 'w') as f:
    f.write(content)

print("Fixed! Verifying...")
# تایید کن که fix شد
!grep -n "num_nodes" /content/HYPER/script/run.py | head -10

Fixed! Verifying...
269:            test_filtered_data = Data(edge_index=full_inference_edges, edge_type=full_inference_etypes, num_nodes=test_data.num_nodes)
275:            test_filtered_data = Data(edge_index=full_inference_edges, edge_type=full_inference_etypes, num_nodes=test_data.num_nodes)
281:                num_nodes=valid_data.num_nodes
285:        filtered_data = Data(edge_index=dataset._data.target_edge_index, edge_type=dataset._data.target_edge_type, num_nodes=dataset[0].num_nodes)


## JF-25


In [ ]:
# Cell 6
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive.yaml \
    --dataset JF-25 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
05:35:38   Random seed: 1024
05:35:38   Config file: config/inference/HYPER_text_inductive.yaml
05:35:38   {'checkpoint': None,
 'dataset': {'class': 'JF-25', 'root': '~/HYPER/kg-datasets/', 'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'gated',
                              'hidden_dims': [64, 

## JF-50

In [ ]:
# Cell 7
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive.yaml \
    --dataset JF-50 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
10:16:38   Random seed: 1024
10:16:38   Config file: config/inference/HYPER_text_inductive.yaml
10:16:38   {'checkpoint': None,
 'dataset': {'class': 'JF-50', 'root': '~/HYPER/kg-datasets/', 'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'gated',
                              'hidden_dims': [64, 

##JF-75

In [11]:
# Cell 8
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive_additive.yaml \
    --dataset JF-75 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
17:56:00   Random seed: 1024
17:56:00   Config file: config/inference/HYPER_text_inductive_additive.yaml
17:56:00   {'checkpoint': None,
 'dataset': {'class': 'JF-75', 'root': '~/HYPER/kg-datasets/', 'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'additive',
                              'hidden_

## JF-100


In [ ]:
# Cell 8
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive_additive.yaml \
    --dataset JF-100 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
13:09:15   Random seed: 1024
13:09:15   Config file: config/inference/HYPER_text_inductive_additive.yaml
13:09:15   {'checkpoint': None,
 'dataset': {'class': 'JF-100',
             'root': '~/HYPER/kg-datasets/',
             'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'additive',
           

In [ ]:
# Cell 8
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive.yaml \
    --dataset JF-100 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
10:30:46   Random seed: 1024
10:30:46   Config file: config/inference/HYPER_text_inductive.yaml
10:30:46   {'checkpoint': None,
 'dataset': {'class': 'JF-100',
             'root': '~/HYPER/kg-datasets/',
             'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'gated',
                       

## WD-25

In [9]:
# Cell 9
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive_additive.yaml \
    --dataset WD-25 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/WD/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/WD/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
17:45:13   Random seed: 1024
17:45:13   Config file: config/inference/HYPER_text_inductive_additive.yaml
17:45:13   {'checkpoint': None,
 'dataset': {'class': 'WD-25', 'root': '~/HYPER/kg-datasets/', 'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'additive',
                              'hidden_

## WP-50

In [ ]:
# Cell 9
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive_additive.yaml \
    --dataset WP-25 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/WP/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/WP/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
13:17:49   Random seed: 1024
13:17:49   Config file: config/inference/HYPER_text_inductive_additive.yaml
13:17:49   {'checkpoint': None,
 'dataset': {'class': 'WP-25', 'root': '~/HYPER/kg-datasets/', 'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'additive',
                              'hidden_

## WP-100


In [31]:
# Cell 8
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive_additive.yaml \
    --dataset WP-100 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/WP/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/WP/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
19:31:28   Random seed: 1024
19:31:28   Config file: config/inference/HYPER_text_inductive_additive.yaml
19:31:28   {'checkpoint': None,
 'dataset': {'class': 'WP-100',
             'root': '~/HYPER/kg-datasets/',
             'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'additive',
           

In [ ]:
# Cell 8
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive_additive.yaml \
    --dataset WP-50 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/WP/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/WP/relation_text.json

/content/HYPER
Neptune project not specified. Will proceed with normal logging
19:36:23   Random seed: 1024
19:36:23   Config file: config/inference/HYPER_text_inductive_additive.yaml
19:36:23   {'checkpoint': None,
 'dataset': {'class': 'WP-50', 'root': '~/HYPER/kg-datasets/', 'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'additive',
                              'hidden_

In [ ]:
!zip -r /content/hyper.zip /content/HYPER

  adding: content/HYPER/ (stored 0%)
  adding: content/HYPER/hyper/ (stored 0%)
  adding: content/HYPER/hyper/util.py (deflated 68%)
  adding: content/HYPER/hyper/__pycache__/ (stored 0%)
  adding: content/HYPER/hyper/__pycache__/tasks.cpython-313.pyc (deflated 50%)
  adding: content/HYPER/hyper/__pycache__/datasets.cpython-313.pyc (deflated 69%)
  adding: content/HYPER/hyper/__pycache__/layers.cpython-313.pyc (deflated 50%)
  adding: content/HYPER/hyper/__pycache__/util.cpython-313.pyc (deflated 52%)
  adding: content/HYPER/hyper/__pycache__/models.cpython-313.pyc (deflated 55%)
  adding: content/HYPER/hyper/layers.py (deflated 74%)
  adding: content/HYPER/hyper/tasks.py (deflated 69%)
  adding: content/HYPER/hyper/datasets.py (deflated 85%)
  adding: content/HYPER/hyper/rspmm/ (stored 0%)
  adding: content/HYPER/hyper/rspmm/__pycache__/ (stored 0%)
  adding: content/HYPER/hyper/rspmm/__pycache__/triton_rspmm.cpython-313.pyc (deflated 66%)
  adding: content/HYPER/hyper/rspmm/triton_rs

## Without G-rel


In [27]:
# Cell 8
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive.yaml \
    --dataset JF-75 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/relation_text.json \
     --num_relations null

/content/HYPER
Neptune project not specified. Will proceed with normal logging
18:48:15   Random seed: 1024
18:48:15   Config file: config/inference/HYPER_text_inductive.yaml
18:48:15   {'checkpoint': None,
 'dataset': {'class': 'JF-75', 'root': '~/HYPER/kg-datasets/', 'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'gated',
                              'hidden_dims': [64, 

In [28]:
# Cell 8
%cd /content/HYPER
!python script/run_text.py -c config/inference/HYPER_text_inductive.yaml \
    --dataset WD-25 --version null --epochs 10 --bpe null --gpus [0] --ckpt null \
    --ent_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/entity_text.json --rel_text_json /content/HYPER/hypergraph_dataset/Multimodal/JFK/relation_text.json \
     --num_relations null

/content/HYPER
Neptune project not specified. Will proceed with normal logging
18:56:44   Random seed: 1024
18:56:44   Config file: config/inference/HYPER_text_inductive.yaml
18:56:44   {'checkpoint': None,
 'dataset': {'class': 'WD-25', 'root': '~/HYPER/kg-datasets/', 'version': None},
 'model': {'class': 'TextHYPER',
           'entity_model': {'aggregate_func': 'sum',
                            'class': 'TextEntityHCNet',
                            'fusion_type': 'gated',
                            'hidden_dims': [64, 64, 64, 64, 64, 64],
                            'input_dim': 64,
                            'short_cut': True,
                            'text_dim': 384,
                            'text_dropout': 0.1,
                            'use_triton': True},
           'relation_model': {'aggregate_func': 'sum',
                              'class': 'TextRelHCNet',
                              'fusion_type': 'gated',
                              'hidden_dims': [64, 